# Analysis: Block vs Other_Areas for Training Schema Design

**Objective**: Analyze the `block` and `other_areas` columns to determine the best approach for training the T5 location extraction model.

**Key Questions**:
1. How sparse are `block` and `other_areas` columns?
2. How often does `other_areas` duplicate or variant-spell the `block` name?
3. What types of information are in `other_areas` (blocks, forests, landmarks, etc.)?
4. Should we merge them into a single `other_location` field for training?

## 1. Load and Inspect Data

In [14]:
import pandas as pd
import numpy as np
from collections import Counter
import re

# Load the full clean dataset (has date, incident_number, and all location fields)
location_data = pd.read_csv('../../data/satp_clean.csv', dtype=str)

print(f"Total records: {len(location_data):,}")
print(f"\nColumns: {location_data.columns.tolist()}")

# Verify we have the key columns for analysis and Telangana correction
required_cols = ['date', 'state', 'district', 'block', 
                 'village_name', 'other_areas', 'incident_summary']
missing_cols = [col for col in required_cols if col not in location_data.columns]

if missing_cols:
    print(f"\n⚠️  Missing columns: {missing_cols}")
else:
    print(f"\n✅ All required columns present (including 'date' for Telangana correction)")

print(f"\nFirst few rows:")
location_data.head()

Total records: 9,921

Columns: ['incident_number', 'state', 'district', 'block', 'village_name', 'other_areas', 'constituency', 'latitude', 'longitude', 'year', 'date', 'perpetrator', 'first_action', 'second_action', 'third_action', 'armed_assault', 'arrest', 'bombing', 'infrastructure', 'surrender', 'seizure', 'abduction', 'action_successful', 'first_target', 'second_target', 'civilians', 'maoist', 'government_officials', 'security', 'private_property', 'mining_company', 'ngos', 'government_infrastructure', 'non_maoist_armed_group', 'no_target', 'first_civilian_target', 'second_civilian_target', 'former_maoist', 'high_caste_landowner', 'police_informer', 'businessman', 'aspiring_politician', 'other_elite', 'other_civilian', 'total_fatalities', 'govt_official_fatalities', 'civilian_fatalities', 'security_fatalities', 'maoist_fatalities', 'other_armed_grp_fatalities', 'total_injuries', 'govt_official_injuries', 'civilian_injuries', 'security_injuries', 'maoist_injuries', 'non_maoist_arm

,incident_number,state,district,block,village_name,other_areas,constituency,latitude,longitude,year,...,commander_arrests,cadre_arrests,sympathizer_arrests,unknown_arrests,total_surrenders,commander_surrenders,cadre_surrenders,sympathizer_surrenders,unknown_surrenders,incident_summary
0,101010701,Andhra Pradesh,Hyderabad,Gachibowli (Rangareddy),NaN,Cyberabad,Serilingampally,17.4325,78.37180555555555,2007,...,0,0,1,0,0,0,0,0,0,An alleged arms supplier to the Communist Part...
1,101010901,Andhra Pradesh,Nizamabad,NaN,Kamareddy,NaN,Kamareddy,18.320888888888888,78.33713888888889,2009,...,0,0,0,0,1,0,1,0,0,A Kamareddy dalam (squad) member belonging to ...
2,101030601,Andhra Pradesh,Khammam,NaN,Bhadrachalam,NaN,Bhadrachalam,17.668055555555558,80.89686111111112,2006,...,1,0,0,0,0,0,0,0,0,Senior CPI-Maoist 'Polit Bureau' and 'central ...
3,101051602,Andhra Pradesh,Vishakhapatnam,NaN,NaN,Visakha Agency,Rayadurg,17.708777777777776,83.30283333333333,2016,...,0,0,0,0,0,0,0,0,0,A TDP leader and former Sarpanch of Jerrela Gr...
4,101060701,Andhra Pradesh,Visakhapatnam,GK Veedhi,Teegalabanda,Pedalavasa,Rayadurg,17.861861111111114,82.19688888888889,2007,...,0,0,0,0,0,0,0,0,0,The CPI-Maoist cadres blasted coffee pulping u...


## 2. Sparsity Analysis

In [15]:
print("="*80)
print("SPARSITY ANALYSIS: Block vs Other_Areas")
print("="*80)

total = len(location_data)

# Block column analysis
blocks_present = location_data['block'].notna().sum()
blocks_missing = location_data['block'].isna().sum()

print(f"\n📊 BLOCK COLUMN:")
print(f"  Records with block: {blocks_present:,} ({blocks_present/total*100:.1f}%)")
print(f"  Records without block: {blocks_missing:,} ({blocks_missing/total*100:.1f}%)")

# Other_areas column analysis
other_areas_present = location_data['other_areas'].notna().sum()
other_areas_missing = location_data['other_areas'].isna().sum()

print(f"\n📊 OTHER_AREAS COLUMN:")
print(f"  Records with other_areas: {other_areas_present:,} ({other_areas_present/total*100:.1f}%)")
print(f"  Records without other_areas: {other_areas_missing:,} ({other_areas_missing/total*100:.1f}%)")

# Overlap analysis
both_present = ((location_data['block'].notna()) & (location_data['other_areas'].notna())).sum()
only_block = ((location_data['block'].notna()) & (location_data['other_areas'].isna())).sum()
only_other = ((location_data['block'].isna()) & (location_data['other_areas'].notna())).sum()
neither = ((location_data['block'].isna()) & (location_data['other_areas'].isna())).sum()

print(f"\n📊 OVERLAP ANALYSIS:")
print(f"  Both block AND other_areas: {both_present:,} ({both_present/total*100:.1f}%)")
print(f"  Only block: {only_block:,} ({only_block/total*100:.1f}%)")
print(f"  Only other_areas: {only_other:,} ({only_other/total*100:.1f}%)")
print(f"  Neither: {neither:,} ({neither/total*100:.1f}%)")

# Combined potential
either_present = ((location_data['block'].notna()) | (location_data['other_areas'].notna())).sum()
print(f"\n💡 POTENTIAL COMBINED FIELD:")
print(f"  Records with block OR other_areas: {either_present:,} ({either_present/total*100:.1f}%)")
print(f"  ↑ This is what we'd have if we merge into 'other_location'")

SPARSITY ANALYSIS: Block vs Other_Areas

📊 BLOCK COLUMN:
  Records with block: 5,360 (54.0%)
  Records without block: 4,561 (46.0%)

📊 OTHER_AREAS COLUMN:
  Records with other_areas: 4,026 (40.6%)
  Records without other_areas: 5,895 (59.4%)

📊 OVERLAP ANALYSIS:
  Both block AND other_areas: 2,663 (26.8%)
  Only block: 2,697 (27.2%)
  Only other_areas: 1,363 (13.7%)
  Neither: 3,198 (32.2%)

💡 POTENTIAL COMBINED FIELD:
  Records with block OR other_areas: 6,723 (67.8%)
  ↑ This is what we'd have if we merge into 'other_location'


## 3. Similarity Analysis: Does other_areas Duplicate block?

In [16]:
print("="*80)
print("SIMILARITY ANALYSIS: Block vs Other_Areas")
print("="*80)

# Filter to records with both fields
both_fields = location_data[(location_data['block'].notna()) & (location_data['other_areas'].notna())].copy()
print(f"\nAnalyzing {len(both_fields):,} records with both block and other_areas...\n")

# Exact match (case-insensitive)
both_fields['exact_match'] = both_fields.apply(
    lambda row: str(row['block']).strip().lower() == str(row['other_areas']).strip().lower(),
    axis=1
)
exact_matches = both_fields['exact_match'].sum()

print(f"🔍 EXACT MATCHES (case-insensitive):")
print(f"  {exact_matches:,} / {len(both_fields):,} ({exact_matches/len(both_fields)*100:.1f}%)")
print(f"  → other_areas is identical to block")

# Substring match (other_areas contains block or vice versa)
both_fields['substring_match'] = both_fields.apply(
    lambda row: (str(row['block']).strip().lower() in str(row['other_areas']).strip().lower()) or
                (str(row['other_areas']).strip().lower() in str(row['block']).strip().lower()),
    axis=1
)
substring_matches = both_fields['substring_match'].sum()

print(f"\n🔍 SUBSTRING MATCHES:")
print(f"  {substring_matches:,} / {len(both_fields):,} ({substring_matches/len(both_fields)*100:.1f}%)")
print(f"  → other_areas contains block name (or vice versa)")

# Different content
different = len(both_fields) - substring_matches
print(f"\n🔍 DIFFERENT CONTENT:")
print(f"  {different:,} / {len(both_fields):,} ({different/len(both_fields)*100:.1f}%)")
print(f"  → other_areas provides distinct information")

print(f"\n" + "="*80)
print("INTERPRETATION:")
print("="*80)
if exact_matches / len(both_fields) > 0.5:
    print("⚠️  High redundancy: other_areas often duplicates block")
    print("   → Merging would mostly deduplicate, not add info")
elif substring_matches / len(both_fields) > 0.5:
    print("⚠️  Moderate redundancy: other_areas often contains block name")
    print("   → Merging would add some new info but with overlap")
else:
    print("✅ Low redundancy: other_areas provides distinct information")
    print("   → Merging would significantly enrich the training data")

SIMILARITY ANALYSIS: Block vs Other_Areas

Analyzing 2,663 records with both block and other_areas...

🔍 EXACT MATCHES (case-insensitive):
  360 / 2,663 (13.5%)
  → other_areas is identical to block

🔍 SUBSTRING MATCHES:
  830 / 2,663 (31.2%)
  → other_areas contains block name (or vice versa)

🔍 DIFFERENT CONTENT:
  1,833 / 2,663 (68.8%)
  → other_areas provides distinct information

INTERPRETATION:
✅ Low redundancy: other_areas provides distinct information
   → Merging would significantly enrich the training data


## 4. Content Analysis: What's in other_areas?

In [17]:
print("="*80)
print("CONTENT ANALYSIS: What types of information are in other_areas?")
print("="*80)

# Sample of other_areas values
other_areas_values = location_data[location_data['other_areas'].notna()]['other_areas'].values

print(f"\n📋 SAMPLE VALUES (first 50):")
print("-"*80)
for i, value in enumerate(other_areas_values[:50], 1):
    print(f"{i:2d}. {value}")

# Categorize by keywords
print(f"\n\n🔍 KEYWORD ANALYSIS:")
print("="*80)

keywords = {
    'forest': ['forest', 'reserve', 'sanctuary', 'park', 'wildlife'],
    'police_station': ['police station', 'ps ', ' ps', 'thana'],
    'block_tehsil': ['block', 'tehsil', 'taluk', 'mandal', 'tahsil'],
    'area_region': ['area', 'region', 'zone', 'sector'],
    'geographic': ['hills', 'valley', 'ghat', 'range', 'mountains'],
}

for category, terms in keywords.items():
    pattern = '|'.join(terms)
    matches = location_data[location_data['other_areas'].notna()]['other_areas'].str.contains(
        pattern, case=False, na=False, regex=True
    ).sum()
    pct = matches / len(other_areas_values) * 100 if len(other_areas_values) > 0 else 0
    print(f"  {category.upper().replace('_', ' '):.<30} {matches:>6,} ({pct:>5.1f}%)")

print(f"\nNote: Categories may overlap (e.g., 'forest area')")

CONTENT ANALYSIS: What types of information are in other_areas?

📋 SAMPLE VALUES (first 50):
--------------------------------------------------------------------------------
 1. Cyberabad
 2. Visakha Agency
 3. Pedalavasa
 4. Kakinada
 5. Dummugudem mandal
 6. Makkuva mandal
 7. Wajeedu
 8. Chintapalli mandal
 9. Pullalacheruvu
10. Dummugudem mandal
11. Bhadrachalam
12. Ramakrishnapur Police Station
13. Vararamachandrapuram
14. Ngarajunasagar Project Guest house
15. G. Madugula mandal
16. G. Madugula mandal
17. Garladinne
18. Kuderu
19. Yellandu
20. Machavaram
21. Allapalli Singaram forest
22. Rapthadu
23. Kannvaram
24. G. Madugula mandal
25. Ramagundam
26. Gathumalla
27. Nallamala
28. Agency area
29. Rekulapally
30. Koyyuru
31. Machavaram
32. Yellandu
33. Venkatapuram
34. Pinapaka
35. Cherla
36. Lovavalasa
37. Dummugudem
38. Begumpet
39. Yellandu
40. Vishakha Agency
41. Lakkavaram forest area
42. Koyyuru Mandal
43. Reddypalem, Karempudi Police Station
44. Pedabayalu
45. Pedabayalu
46.

## 5. Examples: When other_areas ≠ block

In [18]:
print("="*80)
print("EXAMPLES: Cases where other_areas provides DIFFERENT info than block")
print("="*80)

# Get cases where both exist and are different
different_cases = both_fields[~both_fields['substring_match']].copy()

print(f"\nShowing 20 examples from {len(different_cases):,} cases with distinct information:\n")

for idx, (i, row) in enumerate(different_cases.head(20).iterrows(), 1):
    print(f"Example {idx}:")
    print(f"  Block: {row['block']}")
    print(f"  Other_areas: {row['other_areas']}")
    print(f"  District: {row['district']}, State: {row['state']}")
    if pd.notna(row.get('incident_summary')):
        summary = str(row['incident_summary'])[:150]
        print(f"  Incident: {summary}...")
    print()

EXAMPLES: Cases where other_areas provides DIFFERENT info than block

Showing 20 examples from 1,833 cases with distinct information:

Example 1:
  Block: Gachibowli (Rangareddy)
  Other_areas: Cyberabad
  District: Hyderabad, State: Andhra Pradesh
  Incident: An alleged arms supplier to the Communist Party of India-Maoist (CPI-Maoist), identified as Ravi Kumar Chevori, was arrested from Cyberabad near Hyder...

Example 2:
  Block: GK Veedhi
  Other_areas: Pedalavasa
  District: Visakhapatnam, State: Andhra Pradesh
  Incident: The CPI-Maoist cadres blasted coffee pulping units at Teegalabanda and Pedavalasa villages in G.K. Veedhi mandal and took away nearly 350 bags of grad...

Example 3:
  Block: Wazeed
  Other_areas: Wajeedu
  District: Khammam, State: Andhra Pradesh
  Incident: CPI-Maoist cadres trigger a series of landmine blasts targeting a Police party combing the forests of Murmur village in the Wajeedu area of Khammam Di...

Example 4:
  Block: Mandamarri
  Other_areas: Ramakr

## 6. Text Presence Analysis: Are block/other_areas in incident summaries?

In [19]:
print("="*80)
print("TEXT PRESENCE ANALYSIS: How often do block/other_areas appear in incident text?")
print("="*80)

# Filter to records with incident summaries
with_summary = location_data[location_data['incident_summary'].notna()].copy()
print(f"\nAnalyzing {len(with_summary):,} records with incident summaries...\n")

# Block presence in text
with_block = with_summary[with_summary['block'].notna()].copy()
with_block['block_in_text'] = with_block.apply(
    lambda row: str(row['block']).lower() in str(row['incident_summary']).lower(),
    axis=1
)
blocks_in_text = with_block['block_in_text'].sum()

print(f"📊 BLOCK PRESENCE:")
print(f"  Records with block: {len(with_block):,}")
print(f"  Block mentioned in text: {blocks_in_text:,} ({blocks_in_text/len(with_block)*100:.1f}%)")
print(f"  Block NOT in text: {len(with_block)-blocks_in_text:,} ({(len(with_block)-blocks_in_text)/len(with_block)*100:.1f}%)")

# Other_areas presence in text
with_other = with_summary[with_summary['other_areas'].notna()].copy()
with_other['other_in_text'] = with_other.apply(
    lambda row: str(row['other_areas']).lower() in str(row['incident_summary']).lower(),
    axis=1
)
other_in_text = with_other['other_in_text'].sum()

print(f"\n📊 OTHER_AREAS PRESENCE:")
print(f"  Records with other_areas: {len(with_other):,}")
print(f"  Other_areas mentioned in text: {other_in_text:,} ({other_in_text/len(with_other)*100:.1f}%)")
print(f"  Other_areas NOT in text: {len(with_other)-other_in_text:,} ({(len(with_other)-other_in_text)/len(with_other)*100:.1f}%)")

# Combined: Either block or other_areas in text
with_either = with_summary[
    (with_summary['block'].notna()) | (with_summary['other_areas'].notna())
].copy()

def check_either_in_text(row):
    text = str(row['incident_summary']).lower()
    block_in = str(row['block']).lower() in text if pd.notna(row['block']) else False
    other_in = str(row['other_areas']).lower() in text if pd.notna(row['other_areas']) else False
    return block_in or other_in

with_either['either_in_text'] = with_either.apply(check_either_in_text, axis=1)
either_in_text = with_either['either_in_text'].sum()

print(f"\n📊 COMBINED (block OR other_areas):")
print(f"  Records with block or other_areas: {len(with_either):,}")
print(f"  At least one mentioned in text: {either_in_text:,} ({either_in_text/len(with_either)*100:.1f}%)")
print(f"  Neither in text: {len(with_either)-either_in_text:,} ({(len(with_either)-either_in_text)/len(with_either)*100:.1f}%)")

print(f"\n{'='*80}")
print("💡 KEY INSIGHT:")
print("='"*80)
block_pct = blocks_in_text/len(with_block)*100 if len(with_block) > 0 else 0
other_pct = other_in_text/len(with_other)*100 if len(with_other) > 0 else 0
either_pct = either_in_text/len(with_either)*100 if len(with_either) > 0 else 0

if either_pct > block_pct + 5:
    print(f"✅ Merging into 'other_location' would increase text coverage:")
    print(f"   Block only: {block_pct:.1f}% in text")
    print(f"   Combined: {either_pct:.1f}% in text")
    print(f"   → Gain: +{either_pct - block_pct:.1f} percentage points")
else:
    print(f"⚠️  Merging would not significantly improve text coverage:")
    print(f"   Block: {block_pct:.1f}% in text")
    print(f"   Combined: {either_pct:.1f}% in text")

TEXT PRESENCE ANALYSIS: How often do block/other_areas appear in incident text?

Analyzing 9,921 records with incident summaries...

📊 BLOCK PRESENCE:
  Records with block: 5,360
  Block mentioned in text: 2,746 (51.2%)
  Block NOT in text: 2,614 (48.8%)

📊 OTHER_AREAS PRESENCE:
  Records with other_areas: 4,026
  Other_areas mentioned in text: 3,449 (85.7%)
  Other_areas NOT in text: 577 (14.3%)

📊 COMBINED (block OR other_areas):
  Records with block or other_areas: 6,723
  At least one mentioned in text: 4,971 (73.9%)
  Neither in text: 1,752 (26.1%)

💡 KEY INSIGHT:
='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='
✅ Merging into 'other_location' would increase text coverage:
   Block only: 51.2% in text
   Combined: 73.9% in text
   → Gain: +22.7 percentage points


## 7. Training Data Quality: Extractable vs Non-Extractable

In [20]:
print("="*80)
print("TRAINING DATA QUALITY ASSESSMENT")
print("="*80)

total_records = len(location_data)
with_summary = location_data['incident_summary'].notna().sum()

print(f"\nTotal dataset: {total_records:,} records")
print(f"Records with incident summaries: {with_summary:,} ({with_summary/total_records*100:.1f}%)\n")

# Scenario 1: Current approach (block only)
print("="*80)
print("SCENARIO 1: Current Approach (block field only)")
print("="*80)

block_records = location_data['block'].notna().sum()
block_extractable = blocks_in_text if 'blocks_in_text' in locals() else 0
block_not_extractable = block_records - block_extractable

print(f"\nTraining labels with blocks: {block_records:,} ({block_records/total_records*100:.1f}%)")
print(f"  ✓ Extractable (in text): {block_extractable:,} ({block_extractable/total_records*100:.1f}%)")
print(f"  ✗ Not extractable (not in text): {block_not_extractable:,} ({block_not_extractable/total_records*100:.1f}%)")
print(f"  → Training signal quality: {block_extractable/block_records*100 if block_records > 0 else 0:.1f}% clean")

# Scenario 2: Merge into other_location
print(f"\n{'='*80}")
print("SCENARIO 2: Proposed Approach (merge block + other_areas → 'other_location')")
print("="*80)

either_records = ((location_data['block'].notna()) | (location_data['other_areas'].notna())).sum()
either_extractable = either_in_text if 'either_in_text' in locals() else 0
either_not_extractable = either_records - either_extractable

print(f"\nTraining labels with other_location: {either_records:,} ({either_records/total_records*100:.1f}%)")
print(f"  ✓ Extractable (in text): {either_extractable:,} ({either_extractable/total_records*100:.1f}%)")
print(f"  ✗ Not extractable (not in text): {either_not_extractable:,} ({either_not_extractable/total_records*100:.1f}%)")
print(f"  → Training signal quality: {either_extractable/either_records*100 if either_records > 0 else 0:.1f}% clean")

# Comparison
print(f"\n{'='*80}")
print("COMPARISON")
print("="*80)

gain_records = either_records - block_records
gain_extractable = either_extractable - block_extractable

print(f"\n📈 Gain from merging:")
print(f"  Additional training examples: +{gain_records:,} ({gain_records/block_records*100 if block_records > 0 else 0:.1f}% increase)")
print(f"  Additional extractable examples: +{gain_extractable:,}")

block_quality = block_extractable/block_records*100 if block_records > 0 else 0
either_quality = either_extractable/either_records*100 if either_records > 0 else 0
quality_change = either_quality - block_quality

print(f"\n📊 Training signal quality:")
if quality_change > 0:
    print(f"  ✅ Quality IMPROVES: {block_quality:.1f}% → {either_quality:.1f}% (+{quality_change:.1f}pp)")
elif quality_change < -2:
    print(f"  ⚠️  Quality DECREASES: {block_quality:.1f}% → {either_quality:.1f}% ({quality_change:.1f}pp)")
else:
    print(f"  ➡️  Quality STABLE: {block_quality:.1f}% → {either_quality:.1f}% ({quality_change:+.1f}pp)")

TRAINING DATA QUALITY ASSESSMENT

Total dataset: 9,921 records
Records with incident summaries: 9,921 (100.0%)

SCENARIO 1: Current Approach (block field only)

Training labels with blocks: 5,360 (54.0%)
  ✓ Extractable (in text): 2,746 (27.7%)
  ✗ Not extractable (not in text): 2,614 (26.3%)
  → Training signal quality: 51.2% clean

SCENARIO 2: Proposed Approach (merge block + other_areas → 'other_location')

Training labels with other_location: 6,723 (67.8%)
  ✓ Extractable (in text): 4,971 (50.1%)
  ✗ Not extractable (not in text): 1,752 (17.7%)
  → Training signal quality: 73.9% clean

COMPARISON

📈 Gain from merging:
  Additional training examples: +1,363 (25.4% increase)
  Additional extractable examples: +2,225

📊 Training signal quality:
  ✅ Quality IMPROVES: 51.2% → 73.9% (+22.7pp)


## 8. Filtered and Merged Approach

**Strategy**: Instead of A/B testing, implement a single unified approach:
1. Filter blocks to only those mentioned in incident text
2. Remove block name duplicates from other_areas  
3. Merge filtered blocks + deduplicated other_areas → `other_location`
4. Result: High-quality training data with maximum coverage

In [21]:
print("="*80)
print("STEP 1: Filter blocks to only those mentioned in text")
print("="*80)

# Create working copy
location_cleaned = location_data.copy()

# Function to check if block is in text
def block_mentioned_in_text(row):
    if pd.isna(row['block']) or pd.isna(row['incident_summary']):
        return False
    return str(row['block']).lower() in str(row['incident_summary']).lower()

# Mark blocks that appear in text
location_cleaned['block_in_text'] = location_cleaned.apply(block_mentioned_in_text, axis=1)

# Filter: keep only blocks mentioned in text
blocks_to_remove = location_cleaned['block'].notna() & ~location_cleaned['block_in_text']
blocks_removed_count = blocks_to_remove.sum()

print(f"\n📊 BLOCK FILTERING:")
print(f"  Original blocks: {location_cleaned['block'].notna().sum():,}")
print(f"  Blocks mentioned in text: {location_cleaned['block_in_text'].sum():,}")
print(f"  Blocks to remove (not in text): {blocks_removed_count:,}")

# Set non-mentioned blocks to None
location_cleaned.loc[blocks_to_remove, 'block'] = None

print(f"  ✓ Filtered blocks: {location_cleaned['block'].notna().sum():,}")
print(f"  Training signal quality: 100.0% (all blocks now in text)")

# Show examples of removed blocks
print(f"\n📋 EXAMPLES OF REMOVED BLOCKS (not in text):")
print("-"*80)
examples = location_data[blocks_to_remove].head(5)
for idx, (i, row) in enumerate(examples.iterrows(), 1):
    print(f"\n{idx}. Block '{row['block']}' removed")
    print(f"   District: {row['district']}, State: {row['state']}")
    if pd.notna(row['incident_summary']):
        print(f"   Incident: {str(row['incident_summary'])[:120]}...")
print("\n" + "="*80)

STEP 1: Filter blocks to only those mentioned in text

📊 BLOCK FILTERING:
  Original blocks: 5,360
  Blocks mentioned in text: 2,746
  Blocks to remove (not in text): 2,614
  ✓ Filtered blocks: 2,746
  Training signal quality: 100.0% (all blocks now in text)

📋 EXAMPLES OF REMOVED BLOCKS (not in text):
--------------------------------------------------------------------------------

1. Block 'Gachibowli (Rangareddy)' removed
   District: Hyderabad, State: Andhra Pradesh
   Incident: An alleged arms supplier to the Communist Party of India-Maoist (CPI-Maoist), identified as Ravi Kumar Chevori, was arre...

2. Block 'GK Veedhi' removed
   District: Visakhapatnam, State: Andhra Pradesh
   Incident: The CPI-Maoist cadres blasted coffee pulping units at Teegalabanda and Pedavalasa villages in G.K. Veedhi mandal and too...

3. Block 'Chintapalli' removed
   District: Visakhapatnam, State: Andhra Pradesh
   Incident: Around 50 Maoists belonging to the Korukonda Dalam and their sympathisers of

In [22]:
print("="*80)
print("STEP 2: Deduplicate other_areas (remove block names)")
print("="*80)

def remove_block_from_other_areas(row):
    """Remove block name from other_areas if it's a duplicate, but preserve additional info."""
    if pd.isna(row['other_areas']):
        return row['other_areas']
    
    if pd.isna(row['block']):
        return row['other_areas']
    
    block = str(row['block']).strip()
    block_lower = block.lower()
    other = str(row['other_areas']).strip()
    other_lower = other.lower()
    
    # Case 1: Exact match (case-insensitive)
    if block_lower == other_lower:
        return None  # Complete duplicate, remove
    
    # Case 2: other_areas is comma-separated list
    if ',' in other:
        components = [c.strip() for c in other.split(',')]
        # Remove components that exactly match the block or contain it as substring
        filtered = []
        for c in components:
            c_lower = c.lower()
            # Keep component if it's not the block and doesn't just add a descriptor to block
            if c_lower != block_lower and not (block_lower in c_lower and len(c_lower) < len(block_lower) + 20):
                filtered.append(c)
        
        # Return filtered list or None if empty
        if filtered:
            return ', '.join(filtered)
        else:
            return None
    
    # Case 3: Single value - check if it's just the block or block + minor variant
    if block_lower in other_lower:
        # If other_areas is just block + short descriptor (e.g., "Makkuva Block"), remove it
        if len(other_lower) < len(block_lower) + 10:
            return None
        # Otherwise keep it (e.g., "Makkuva, Nilgiri Hills" should become just this value)
        # This case shouldn't happen if comma-separated, but keep as fallback
        return other
    
    # Case 4: block contains other_areas (reverse substring)
    if other_lower in block_lower and len(other_lower) < len(block_lower) + 10:
        return None  # Reverse duplicate, remove
    
    # Case 5: Different content - keep it
    return other

# Apply deduplication
location_cleaned['other_areas_original'] = location_cleaned['other_areas'].copy()
location_cleaned['other_areas'] = location_cleaned.apply(remove_block_from_other_areas, axis=1)

# Calculate impact
other_removed = (location_cleaned['other_areas_original'].notna() & location_cleaned['other_areas'].isna()).sum()
other_kept = location_cleaned['other_areas'].notna().sum()

print(f"\n📊 OTHER_AREAS DEDUPLICATION:")
print(f"  Original other_areas: {location_cleaned['other_areas_original'].notna().sum():,}")
print(f"  Duplicates removed: {other_removed:,}")
print(f"  Unique other_areas kept: {other_kept:,}")

# Show examples of removed duplicates
print(f"\n📋 EXAMPLES OF REMOVED DUPLICATES:")
print("-"*80)
removed_mask = (location_cleaned['other_areas_original'].notna() & location_cleaned['other_areas'].isna())
examples = location_cleaned[removed_mask].head(10)
for idx, (i, row) in enumerate(examples.iterrows(), 1):
    print(f"\n{idx}. Removed: '{row['other_areas_original']}'")
    print(f"   Reason: Duplicate/variant of block '{row['block']}'")
    print(f"   District: {row['district']}, State: {row['state']}")

# Show examples of kept other_areas (distinct from block)
print(f"\n\n📋 EXAMPLES OF KEPT OTHER_AREAS (distinct from block):")
print("-"*80)
kept_mask = (location_cleaned['block'].notna() & location_cleaned['other_areas'].notna())
examples = location_cleaned[kept_mask].head(10)
for idx, (i, row) in enumerate(examples.iterrows(), 1):
    print(f"\n{idx}. Block: '{row['block']}'")
    print(f"   Other_areas: '{row['other_areas']}' (KEPT - distinct info)")
    print(f"   District: {row['district']}, State: {row['state']}")

print("\n" + "="*80)

STEP 2: Deduplicate other_areas (remove block names)

📊 OTHER_AREAS DEDUPLICATION:
  Original other_areas: 4,026
  Duplicates removed: 458
  Unique other_areas kept: 3,568

📋 EXAMPLES OF REMOVED DUPLICATES:
--------------------------------------------------------------------------------

1. Removed: 'Kakinada'
   Reason: Duplicate/variant of block 'Kakinada'
   District: East Godavari, State: Andhra Pradesh

2. Removed: 'Makkuva mandal'
   Reason: Duplicate/variant of block 'Makkuva'
   District: Vizianagaram, State: Andhra Pradesh

3. Removed: 'Chintapalli mandal'
   Reason: Duplicate/variant of block 'Chintapalli'
   District: Visakhapatnam, State: Andhra Pradesh

4. Removed: 'Dummugudem mandal'
   Reason: Duplicate/variant of block 'Dummugudem'
   District: Khammam, State: Andhra Pradesh

5. Removed: 'Vararamachandrapuram'
   Reason: Duplicate/variant of block 'Vararamachandrapuram'
   District: Khammam, State: Andhra Pradesh

6. Removed: 'G. Madugula mandal'
   Reason: Duplicate/va

In [23]:
print("="*80)
print("STEP 3: Merge filtered blocks + deduplicated other_areas → 'other_location'")
print("="*80)

def create_other_location(row):
    """Merge block and other_areas into single other_location field."""
    parts = []
    
    # Add block if present
    if pd.notna(row['block']):
        parts.append(str(row['block']).strip())
    
    # Add other_areas if present and different
    if pd.notna(row['other_areas']):
        parts.append(str(row['other_areas']).strip())
    
    # Return merged or None
    if parts:
        return ', '.join(parts)
    else:
        return None

# Create the merged field
location_cleaned['other_location'] = location_cleaned.apply(create_other_location, axis=1)

# Analyze the result
other_location_count = location_cleaned['other_location'].notna().sum()
both_components = ((location_cleaned['block'].notna()) & (location_cleaned['other_areas'].notna())).sum()
only_block = ((location_cleaned['block'].notna()) & (location_cleaned['other_areas'].isna())).sum()
only_other = ((location_cleaned['block'].isna()) & (location_cleaned['other_areas'].notna())).sum()

print(f"\n📊 MERGED FIELD COMPOSITION:")
print(f"  Total records with other_location: {other_location_count:,} ({other_location_count/len(location_cleaned)*100:.1f}%)")
print(f"    - Block + other_areas: {both_components:,} ({both_components/other_location_count*100:.1f}%)")
print(f"    - Block only: {only_block:,} ({only_block/other_location_count*100:.1f}%)")
print(f"    - Other_areas only: {only_other:,} ({only_other/other_location_count*100:.1f}%)")

# Check text presence
def check_other_location_in_text(row):
    if pd.isna(row['other_location']) or pd.isna(row['incident_summary']):
        return False
    
    # Check if ANY component appears in text
    components = str(row['other_location']).split(', ')
    text = str(row['incident_summary']).lower()
    
    for component in components:
        if component.strip().lower() in text:
            return True
    return False

location_cleaned['other_location_in_text'] = location_cleaned.apply(check_other_location_in_text, axis=1)
other_loc_in_text = location_cleaned['other_location_in_text'].sum()

print(f"\n📊 TEXT PRESENCE:")
print(f"  Other_location mentioned in text: {other_loc_in_text:,} ({other_loc_in_text/other_location_count*100:.1f}%)")
print(f"  Other_location NOT in text: {other_location_count - other_loc_in_text:,} ({(other_location_count - other_loc_in_text)/other_location_count*100:.1f}%)")

print(f"\n📋 EXAMPLES OF MERGED OTHER_LOCATION:")
print("-"*80)
examples = location_cleaned[location_cleaned['other_location'].notna()].head(15)
for idx, (i, row) in enumerate(examples.iterrows(), 1):
    print(f"\n{idx}. Other_location: '{row['other_location']}'")
    if pd.notna(row.get('block')) and pd.notna(row.get('other_areas')):
        print(f"   Components: block='{row['block']}', other_areas='{row['other_areas']}'")
    print(f"   District: {row['district']}, State: {row['state']}")
    print(f"   In text: {'✓' if row['other_location_in_text'] else '✗'}")

print("\n" + "="*80)

STEP 3: Merge filtered blocks + deduplicated other_areas → 'other_location'

📊 MERGED FIELD COMPOSITION:
  Total records with other_location: 5,388 (54.3%)
    - Block + other_areas: 926 (17.2%)
    - Block only: 1,820 (33.8%)
    - Other_areas only: 2,642 (49.0%)

📊 TEXT PRESENCE:
  Other_location mentioned in text: 5,032 (93.4%)
  Other_location NOT in text: 356 (6.6%)

📋 EXAMPLES OF MERGED OTHER_LOCATION:
--------------------------------------------------------------------------------

1. Other_location: 'Cyberabad'
   District: Hyderabad, State: Andhra Pradesh
   In text: ✓

2. Other_location: 'Visakha Agency'
   District: Vishakhapatnam, State: Andhra Pradesh
   In text: ✓

3. Other_location: 'Pedalavasa'
   District: Visakhapatnam, State: Andhra Pradesh
   In text: ✗

4. Other_location: 'Kakinada'
   District: East Godavari, State: Andhra Pradesh
   In text: ✓

5. Other_location: 'Koyyuru'
   District: Visakhapatnam, State: Andhra Pradesh
   In text: ✓

6. Other_location: 'Dummug

In [24]:
print("="*80)
print("STEP 4: Final Comparison - Original vs Cleaned Approach")
print("="*80)

print("\n" + "="*80)
print("ORIGINAL APPROACH (block field with all values)")
print("="*80)
original_blocks = location_data['block'].notna().sum()
original_in_text = 2746  # From earlier analysis
print(f"\nTraining examples: {original_blocks:,}")
print(f"  ✓ In text: {original_in_text:,} ({original_in_text/original_blocks*100:.1f}%)")
print(f"  ✗ Not in text: {original_blocks - original_in_text:,} ({(original_blocks - original_in_text)/original_blocks*100:.1f}%)")
print(f"Training signal quality: 51.2%")

print("\n" + "="*80)
print("CLEANED APPROACH (filtered + merged → 'other_location')")
print("="*80)
cleaned_other_loc = location_cleaned['other_location'].notna().sum()
cleaned_in_text = location_cleaned['other_location_in_text'].sum()
print(f"\nTraining examples: {cleaned_other_loc:,}")
print(f"  ✓ In text: {cleaned_in_text:,} ({cleaned_in_text/cleaned_other_loc*100:.1f}%)")
print(f"  ✗ Not in text: {cleaned_other_loc - cleaned_in_text:,} ({(cleaned_other_loc - cleaned_in_text)/cleaned_other_loc*100:.1f}%)")
print(f"Training signal quality: {cleaned_in_text/cleaned_other_loc*100:.1f}%")

print("\n" + "="*80)
print("COMPARISON SUMMARY")
print("="*80)
example_gain = cleaned_other_loc - original_blocks
extractable_gain = cleaned_in_text - original_in_text
quality_gain = (cleaned_in_text/cleaned_other_loc - original_in_text/original_blocks) * 100

print(f"\n📈 GAINS:")
print(f"  Additional training examples: {example_gain:,} ({example_gain/original_blocks*100:+.1f}%)")
print(f"  Additional extractable examples: {extractable_gain:,} ({extractable_gain/original_in_text*100:+.1f}%)")
print(f"  Training signal quality improvement: {quality_gain:+.1f} percentage points")

print(f"\n✅ BENEFITS:")
print(f"  1. More training data: {cleaned_other_loc:,} vs {original_blocks:,} examples")
print(f"  2. Better quality: {cleaned_in_text/cleaned_other_loc*100:.1f}% vs 51.2% extractable")
print(f"  3. No redundancy: Duplicates removed between block and other_areas")
print(f"  4. Richer context: Includes police stations, forests, landmarks")
print(f"  5. Flexible extraction: Not limited to administrative blocks")

print("\n" + "="*80)

STEP 4: Final Comparison - Original vs Cleaned Approach

ORIGINAL APPROACH (block field with all values)

Training examples: 5,360
  ✓ In text: 2,746 (51.2%)
  ✗ Not in text: 2,614 (48.8%)
Training signal quality: 51.2%

CLEANED APPROACH (filtered + merged → 'other_location')

Training examples: 5,388
  ✓ In text: 5,032 (93.4%)
  ✗ Not in text: 356 (6.6%)
Training signal quality: 93.4%

COMPARISON SUMMARY

📈 GAINS:
  Additional training examples: 28 (+0.5%)
  Additional extractable examples: 2,286 (+83.2%)
  Training signal quality improvement: +42.2 percentage points

✅ BENEFITS:
  1. More training data: 5,388 vs 5,360 examples
  2. Better quality: 93.4% vs 51.2% extractable
  3. No redundancy: Duplicates removed between block and other_areas
  4. Richer context: Includes police stations, forests, landmarks
  5. Flexible extraction: Not limited to administrative blocks



## 9. Deduplicate Village Names from Other_Locations

**Rationale**: After merging `block` and `other_areas` into `other_locations`, we need one final deduplication step. The `village_name` field sometimes appears in `other_locations` as well, creating redundancy.

**Deduplication strategy**:
1. **Exact match**: If `other_locations` = `village_name`, remove entire `other_locations` field
2. **Partial match**: If `village_name` appears in a comma-separated list in `other_locations`, remove only that component
3. **Preserve distinct info**: Keep `other_locations` values that provide additional context beyond the village name

In [25]:
print("="*80)
print("STEP 9: Deduplicate Village Names from Other_Locations")
print("="*80)

# Create output dataframe with all original columns plus other_locations
output_df = location_data.copy()

# Add the cleaned other_location column (from Step 9)
output_df['other_locations'] = location_cleaned['other_location']

# Store original for comparison
output_df['other_locations_original'] = output_df['other_locations'].copy()

def remove_village_from_other_locations(row):
    """Remove village name from other_locations if it's a duplicate."""
    if pd.isna(row['other_locations']) or pd.isna(row['village_name']):
        return row['other_locations']
    
    village = str(row['village_name']).strip()
    village_lower = village.lower()
    other_loc = str(row['other_locations']).strip()
    other_loc_lower = other_loc.lower()
    
    # Case 1: Exact match
    if village_lower == other_loc_lower:
        return None
    
    # Case 2: Comma-separated list
    if ',' in other_loc:
        components = [c.strip() for c in other_loc.split(',')]
        filtered = []
        for c in components:
            c_lower = c.lower()
            # Keep component if it doesn't match village
            if c_lower != village_lower and not (village_lower in c_lower and len(c_lower) < len(village_lower) + 10):
                filtered.append(c)
        return ', '.join(filtered) if filtered else None
    
    # Case 3: Single value - check if it's just the village or village + minor variant
    if village_lower in other_loc_lower and len(other_loc_lower) < len(village_lower) + 10:
        return None
    
    # Case 4: Reverse substring check
    if other_loc_lower in village_lower and len(other_loc_lower) < len(village_lower) + 10:
        return None
    
    return other_loc

output_df['other_locations'] = output_df.apply(remove_village_from_other_locations, axis=1)

# Report deduplication statistics
village_duplicates_removed = (output_df['other_locations_original'].notna() & output_df['other_locations'].isna()).sum()
village_duplicates_partial = ((output_df['other_locations_original'] != output_df['other_locations']) & 
                               output_df['other_locations_original'].notna() & 
                               output_df['other_locations'].notna()).sum()
total_deduplicated = village_duplicates_removed + village_duplicates_partial

print(f"\n📊 VILLAGE NAME DEDUPLICATION RESULTS:")
print(f"  Records with both village_name and other_locations: {(output_df['village_name'].notna() & output_df['other_locations_original'].notna()).sum():,}")
print(f"  Complete duplicates removed (other_locations = village): {village_duplicates_removed:,}")
print(f"  Partial duplicates (village in list): {village_duplicates_partial:,}")
print(f"  Total records deduplicated: {total_deduplicated:,}")
print(f"  Records with other_locations after deduplication: {output_df['other_locations'].notna().sum():,}")

# Show examples of deduplication
if total_deduplicated > 0:
    print(f"\n📋 EXAMPLES OF VILLAGE NAME DEDUPLICATION:")
    print("-"*80)
    
    # Show complete duplicates
    if village_duplicates_removed > 0:
        print("\n  Complete duplicates removed:")
        removed_mask = (output_df['other_locations_original'].notna() & output_df['other_locations'].isna())
        examples = output_df[removed_mask].head(5)
        for idx, (i, row) in enumerate(examples.iterrows(), 1):
            print(f"    {idx}. Village: '{row['village_name']}' | Other_locations (removed): '{row['other_locations_original']}'")
    
    # Show partial duplicates
    if village_duplicates_partial > 0:
        print("\n  Partial duplicates (village removed from list):")
        partial_mask = ((output_df['other_locations_original'] != output_df['other_locations']) & 
                        output_df['other_locations_original'].notna() & 
                        output_df['other_locations'].notna())
        examples = output_df[partial_mask].head(5)
        for idx, (i, row) in enumerate(examples.iterrows(), 1):
            print(f"    {idx}. Village: '{row['village_name']}'")
            print(f"       Before: '{row['other_locations_original']}'")
            print(f"       After:  '{row['other_locations']}'")

# Drop the comparison column
output_df.drop(columns=['other_locations_original'], inplace=True)

print("\n" + "="*80)
print("✅ Village deduplication complete!")
print("="*80)


STEP 9: Deduplicate Village Names from Other_Locations

📊 VILLAGE NAME DEDUPLICATION RESULTS:
  Records with both village_name and other_locations: 2,860
  Complete duplicates removed (other_locations = village): 325
  Partial duplicates (village in list): 99
  Total records deduplicated: 424
  Records with other_locations after deduplication: 5,063

📋 EXAMPLES OF VILLAGE NAME DEDUPLICATION:
--------------------------------------------------------------------------------

  Complete duplicates removed:
    1. Village: 'Rajavommangi' | Other_locations (removed): 'Rajavommangi'
    2. Village: 'Tenali' | Other_locations (removed): 'Tenali'
    3. Village: 'Paderu Agency area' | Other_locations (removed): 'Agency area'
    4. Village: 'Kakinada' | Other_locations (removed): 'Kakinada'
    5. Village: 'Chilakaluripet' | Other_locations (removed): 'Chilakaluripet'

  Partial duplicates (village removed from list):
    1. Village: 'Karampudi'
       Before: 'Karampudi, Gurazala'
       After

## 10. Export Augmented Dataset

**Final dataset preparation**: Now that we have:
1. Filtered blocks to only those in incident text (Step 9.1)
2. Removed block duplicates from other_areas (Step 9.2)
3. Merged filtered blocks + deduplicated other_areas → `other_locations` (Step 9.3)
4. Removed village name duplicates from other_locations (Step 11)

We're ready to export the clean, augmented dataset with the new `other_locations` field for model training.

In [26]:
print("="*80)
print("STEP 10: Export Augmented Dataset")
print("="*80)

# Filter out rows where district is "99" (missing/unknown district code)
print(f"\n🧹 Filtering out records with district = '99'...")
print(f"   Records before filtering: {len(output_df):,}")
records_with_99 = output_df[output_df['district'] == '99']
print(f"   Records with district='99': {len(records_with_99):,}")

if len(records_with_99) > 0:
    print(f"\n   Examples of records being removed:")
    for idx, (i, row) in enumerate(records_with_99.head(3).iterrows(), 1):
        village = row.get('village_name', 'N/A')
        other_areas = row.get('other_areas', 'N/A')
        # Safely handle NaN values when slicing
        if pd.notna(other_areas) and isinstance(other_areas, str):
            other_areas_display = other_areas[:50] + '...' if len(other_areas) > 50 else other_areas
        else:
            other_areas_display = 'N/A'
        print(f"   {idx}. State: {row['state']}, Village: {village}, Other_areas: {other_areas_display}")

output_df = output_df[output_df['district'] != '99'].copy()
print(f"   Records after filtering: {len(output_df):,}")
print(f"   Records removed: {len(records_with_99):,}")

# Ensure we keep critical columns for model training and geocoding
# Keep: date (for Telangana correction), all location fields, incident_summary
cols_to_keep = [
    'incident_number', 'date', 'state', 'district', 'village_name', 
    'other_locations', 'latitude', 'longitude', 'incident_summary'
]

# Reorder: start with date, then location fields
output_df = output_df[cols_to_keep]

# Normalize date column to YYYY-MM-DD strings for compatibility
if 'date' in output_df.columns:
    print("\n🗓️ Normalizing and filtering dates...")
    print(f"   Records before date filtering: {len(output_df):,}")
    
    # Convert to datetime
    _dt = pd.to_datetime(output_df['date'], errors='coerce', utc=True)
    
    # Filter: keep only records with valid dates >= 2005-01-01 (when database started)
    database_start = pd.to_datetime('2005-01-01', utc=True)
    valid_date_mask = (_dt.notna()) & (_dt >= database_start)
    
    # Count records being removed
    missing_dates = _dt.isna().sum()
    pre_2005_dates = (_dt.notna() & (_dt < database_start)).sum()
    
    if missing_dates > 0 or pre_2005_dates > 0:
        print(f"   Removing records:")
        if missing_dates > 0:
            print(f"     - Missing dates: {missing_dates}")
        if pre_2005_dates > 0:
            print(f"     - Dates before 2005: {pre_2005_dates}")
            # Show the pre-2005 dates
            pre_2005_records = output_df[_dt.notna() & (_dt < database_start)]
            for idx, (i, row) in enumerate(pre_2005_records.iterrows(), 1):
                date_str = _dt.loc[i].strftime('%Y-%m-%d')
                print(f"       {idx}. Date: {date_str}, State: {row.get('state', 'N/A')}, District: {row.get('district', 'N/A')}")
    
    # Apply filter
    output_df = output_df[valid_date_mask].copy()
    _dt = _dt[valid_date_mask]
    
    # Convert to string format
    output_df['date'] = _dt.dt.strftime('%Y-%m-%d')
    
    print(f"   Records after date filtering: {len(output_df):,}")
    print(f"   Date range: {output_df['date'].min()} to {output_df['date'].max()}")
    
    # Show a couple examples
    print("   Examples:")
    for ex in output_df['date'].head(2).tolist():
        print(f"    - {ex}")

# Save to CSV with UTF-8 encoding (with BOM for Excel compatibility)
output_path = '../../data/location_info_augmented.csv'
output_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\n✅ Saved augmented dataset to: {output_path}")
print(f"   Encoding: UTF-8 with BOM (Excel-compatible)")

print(f"\n📊 DATASET SUMMARY:")
print(f"  Total records: {len(output_df):,}")
print(f"  Columns: {len(output_df.columns)}")
print(f"  Column names: {output_df.columns.tolist()}")

print(f"\n📊 KEY COLUMNS:")
print(f"  Records with date: {output_df['date'].notna().sum():,} ({output_df['date'].notna().sum()/len(output_df)*100:.1f}%)")
print(f"  Records with other_locations: {output_df['other_locations'].notna().sum():,} ({output_df['other_locations'].notna().sum()/len(output_df)*100:.1f}%)")
print(f"  Records with village_name: {output_df['village_name'].notna().sum():,} ({output_df['village_name'].notna().sum()/len(output_df)*100:.1f}%)")

print(f"\n📋 SAMPLE RECORDS WITH OTHER_LOCATIONS:")
print("-"*80)
sample = output_df[output_df['other_locations'].notna()].head(10)
for idx, (i, row) in enumerate(sample.iterrows(), 1):
    print(f"\n{idx}. Date: {row.get('date', 'N/A')}")
    print(f"   State: {row['state']}, District: {row['district']}")
    print(f"   Village: {row.get('village_name', 'N/A')}")
    print(f"   Other_locations: {row['other_locations']}")

print(f"\n{'='*80}")
print("✅ READY FOR MODEL TRAINING")
print("="*80)
print("""
This augmented dataset now has:
  ✓ Date column (for Telangana correction in geocoding)
  ✓ Other_locations (filtered + merged block + other_areas)
  ✓ High quality: 73.9% of other_locations appear in incident text
  ✓ No duplicates: Removed block and village name redundancies

Next step: Use this in location_extraction.ipynb for model training
""")

STEP 10: Export Augmented Dataset

🧹 Filtering out records with district = '99'...
   Records before filtering: 9,921
   Records with district='99': 5

   Examples of records being removed:
   1. State: Andhra Pradesh, Village: nan, Other_areas: N/A
   2. State: Jharkhand, Village: Puroshottampur, Other_areas: Daldaliya canal
   3. State: Jharkhand, Village: Chainpur, Other_areas: N/A
   Records after filtering: 9,916
   Records removed: 5

🗓️ Normalizing and filtering dates...
   Records before date filtering: 9,916
   Removing records:
     - Missing dates: 1
     - Dates before 2005: 1
       1. Date: 1940-09-04, State: Andhra Pradesh, District: Visakhapatnam
   Records after date filtering: 9,914
   Date range: 2005-01-03 to 2018-03-08
   Examples:
    - 2007-01-01
    - 2009-01-01

✅ Saved augmented dataset to: ../../data/location_info_augmented.csv
   Encoding: UTF-8 with BOM (Excel-compatible)

📊 DATASET SUMMARY:
  Total records: 9,914
  Columns: 9
  Column names: ['incident_num

In [27]:
output_df.head()

,incident_number,date,state,district,village_name,other_locations,latitude,longitude,incident_summary
0,101010701,2007-01-01,Andhra Pradesh,Hyderabad,NaN,Cyberabad,17.4325,78.37180555555555,An alleged arms supplier to the Communist Part...
1,101010901,2009-01-01,Andhra Pradesh,Nizamabad,Kamareddy,None,18.320888888888888,78.33713888888889,A Kamareddy dalam (squad) member belonging to ...
2,101030601,2006-01-03,Andhra Pradesh,Khammam,Bhadrachalam,None,17.668055555555558,80.89686111111112,Senior CPI-Maoist 'Polit Bureau' and 'central ...
3,101051602,2016-01-05,Andhra Pradesh,Vishakhapatnam,NaN,Visakha Agency,17.708777777777776,83.30283333333333,A TDP leader and former Sarpanch of Jerrela Gr...
4,101060701,2007-01-06,Andhra Pradesh,Visakhapatnam,Teegalabanda,Pedalavasa,17.861861111111114,82.19688888888889,The CPI-Maoist cadres blasted coffee pulping u...
